# AdPilot Egyptian Real Estate Market Intelligence

End-to-end, standalone analysis of Egyptian property listings: data inspection, approved and delegated cleaning decisions, EDA, market tiers, fair-price modeling, explainability, and dashboard-ready exports.

**Important:** This notebook predicts asking-price positioning, not completed transaction value or a certified appraisal. AdPilot schema integration is intentionally postponed.

## 1. Environment and reproducibility

The pipeline preserves raw data, uses deterministic seeds, writes every exclusion to an audit dataset, and keeps the final test set untouched until model selection is complete.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipeline import run_pipeline
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Project root:", ROOT)

## 2. Execute the preparation, EDA, and model pipeline

Set `RETRAIN_MODELS=True` to reproduce training, validation comparison, grouped cross-validation, final refitting, test evaluation, SHAP outputs, and dashboard datasets. Set it to `False` to reuse the bundled model for a faster rerun.

In [ ]:
RETRAIN_MODELS = False
result = run_pipeline(ROOT, retrain=RETRAIN_MODELS)
print("Pipeline completed.")
display(pd.DataFrame(result["comparison"]))

## 3. Prepared analytical dataset

The scope is restricted to sale listings in EGP for apartments, villas, townhouses, duplexes, chalets, and twin houses. Excluded records remain recoverable in the audit files.

In [ ]:
market = pd.read_parquet(ROOT / "data/dashboard/market_listings.parquet")
prep = json.loads((ROOT / "reports/data_preparation_summary.json").read_text(encoding="utf-8"))
print(f"Analytical shape: {market.shape[0]:,} rows × {market.shape[1]} columns")
display(pd.Series(prep, name="value").to_frame())
display(market.head())

### Data-quality and exclusion audit

Every model-scope exclusion has an explicit reason. The raw CSV and the post-duplicate dataset are not overwritten.

In [ ]:
exclusions = pd.read_csv(ROOT / "reports/tables/exclusion_summary.csv")
decisions = pd.read_csv(ROOT / "reports/decision_log.csv")
display(exclusions)
display(decisions.tail(12))

## 4. Feature engineering

Key deployment-safe features include area, bedrooms, bathrooms, coordinates, listing age, amenity and image counts, normalized location fields, property type, furnishing, payment method, completion status, new-city status, and compound status.

Price-derived variables are used only for analysis and tiering—not as model inputs.

In [ ]:
feature_preview = market.copy()
feature_preview["log_price"] = np.log(feature_preview["price_egp"])
feature_preview["bedrooms_per_100_sqm"] = feature_preview["bedrooms_num"] / feature_preview["area_sqm"] * 100
feature_preview["bathrooms_per_100_sqm"] = feature_preview["bathrooms_num"] / feature_preview["area_sqm"] * 100
engineered = [
    "price_per_sqm", "log_price", "bedrooms_per_100_sqm",
    "bathrooms_per_100_sqm", "listing_age_days", "amenities_count",
    "new_city_indicator", "compound_status", "luxury_location_indicator",
    "tier_score", "market_tier"
]
display(feature_preview[engineered].head())

## 5. Exploratory data analysis

The figures below show distributions and segment differences. Each interpretation should be read with sample size and listing-quality limitations in mind.

In [ ]:
summary = pd.read_csv(ROOT / "reports/tables/summary_statistics.csv", index_col=0)
property_summary = pd.read_csv(ROOT / "reports/tables/property_type_summary.csv")
location_summary = pd.read_csv(ROOT / "reports/tables/location_summary.csv")
tier_summary = pd.read_csv(ROOT / "reports/tables/market_tier_summary.csv")
display(summary)
display(property_summary)
display(location_summary.head(20))

### Price distribution

**What it shows:** Asking prices are strongly right-skewed.  
**Why it matters:** Median and log-scale summaries are more representative than the mean alone.  
**Action:** Segment campaigns and budget expectations by property type and location.  
**Caution:** These are asking prices, not transaction prices.

In [ ]:
display(Image(filename=str(ROOT / "reports/figures/price_distribution_log.png")))

### Price per square metre by property type

**Strongest insight:** Villas have the highest observed median EGP/sqm in the cleaned sample.  
**Action:** Use property-type-specific benchmarks rather than one national threshold.  
**Caution:** Location and project mix differ across property types.

In [ ]:
display(Image(filename=str(ROOT / "reports/figures/ppsqm_by_property_type.png")))

### Location intelligence

**Strongest insight:** Sidi Abdel Rahman is the highest-priced town among locations with at least 50 usable records.  
**Action:** Compare locations using both EGP/sqm and sample size.  
**Caution:** Broad marketing regions and formal administrative areas are mixed in the source.

In [ ]:
display(Image(filename=str(ROOT / "reports/figures/median_ppsqm_by_town.png")))
display(Image(filename=str(ROOT / "reports/figures/listing_count_by_region.png")))

## 6. Market-tier design

The tier score combines 60% global EGP/sqm percentile and 40% percentile within town/property type. Segments below 30 records fall back to market-region/property-type percentiles. This preserves absolute market positioning while reducing the chance that every listing in an expensive location is automatically labeled luxury.

In [ ]:
display(tier_summary)
display(Image(filename=str(ROOT / "reports/figures/market_tier_distribution.png")))

## 7. Preliminary model design and leakage controls

- Target: `log(price_egp)`, converted back to EGP for evaluation.
- Grouped split: named submarket/compound, otherwise town + district.
- Excluded predictors: price-derived fields, listing text, IDs, URLs, references, agent/broker/contact identifiers.
- Model selection uses validation results; the test set is evaluated once after final refitting.

In [ ]:
comparison = pd.read_csv(ROOT / "reports/tables/model_comparison.csv")
cv = pd.read_csv(ROOT / "reports/tables/cross_validation_results.csv")
meta = json.loads((ROOT / "models/metadata/model_metadata.json").read_text(encoding="utf-8"))
display(comparison)
display(cv)
print(meta["selection_basis"])

### Model-selection interpretation

Random Forest achieved the lowest validation MAE and RMSE. CatBoost achieved lower validation median absolute error and MAPE, and it handles high-cardinality categorical values natively. CatBoost was therefore selected as the deployment model as a documented business trade-off—not because of test performance.

## 8. Untouched test evaluation

Business-facing errors are reported in EGP. The empirical fair-price interval is calibrated from validation residuals.

In [ ]:
display(pd.Series(meta["test_metrics"], name="test metric").to_frame())
test = pd.read_parquet(ROOT / "reports/tables/test_predictions.parquet")
display(test.sort_values("absolute_error", ascending=False).head(10))
display(Image(filename=str(ROOT / "reports/figures/model_actual_vs_predicted.png")))
display(Image(filename=str(ROOT / "reports/figures/residual_distribution.png")))

## 9. Error analysis by segment

Segment metrics expose where the model is weak. Small groups should not be interpreted as stable performance estimates.

In [ ]:
for name in ["error_by_property_type.csv", "error_by_market_region.csv", "error_by_market_tier.csv", "error_by_town.csv"]:
    print(name)
    display(pd.read_csv(ROOT / "reports/tables" / name).head(20))

## 10. Explainability

SHAP values describe how features influence predictions within this model. They are predictive associations, not causal effects.

In [ ]:
shap_global = pd.read_csv(ROOT / "reports/tables/shap_global_importance.csv")
local = pd.read_csv(ROOT / "reports/tables/example_local_explanation.csv")
display(shap_global.head(20))
display(local.head(15))
display(Image(filename=str(ROOT / "reports/figures/shap_feature_importance.png")))

## 11. Fair-price and opportunity outputs

For each analytical listing, the pipeline exports a predicted fair asking price, validation-calibrated range, under/fair/over classification, model-support count, out-of-distribution flags, and confidence label.

In [ ]:
cols=["title","town","property_type","area_sqm","price_egp","predicted_fair_price","lower_fair_price","upper_fair_price","pricing_status","model_confidence"]
display(market[cols].head(20))
print(market["pricing_status"].value_counts())

## 12. Dashboard and deployment artifacts

The Streamlit application loads precomputed datasets and the exported CatBoost model; it does not retrain during use. Run it from the project root with:

```bash
streamlit run dashboard/app.py
```

In [ ]:
required = [
    ROOT / "dashboard/app.py", ROOT / "models/final_model/catboost_fair_price.cbm",
    ROOT / "data/dashboard/market_listings.parquet", ROOT / "data/dashboard/pricing_opportunities.parquet",
    ROOT / "requirements.txt", ROOT / "README.md"
]
for p in required:
    print("OK" if p.exists() else "MISSING", p.relative_to(ROOT))

## 13. Executive conclusion

The system provides a defensible first-pass asking-price benchmark and market-positioning tool for Egyptian residential sale listings. Its strongest use is comparative market intelligence, campaign segmentation, and anomaly review. It should not replace professional appraisal, verified transaction data, or manual review of unusual listings.

### Limitations and next steps

1. Acquire completed-transaction or reservation data to reduce asking-price bias.
2. Add a verified developer/project registry before developer scoring.
3. Improve geographic entity resolution and add approved coordinates/mappings.
4. Monitor drift and retrain as market conditions change.
5. Collect richer payment-plan, delivery, finishing, floor, and project-age fields.